# Cross-Asset Seasonality — BQuant (matplotlib), real-roll-calendar

Roll-adjusted futures / FX / swap spreads. Metric by asset type:
- **with price** (ES1, VG1, TY1, RX1, CL1, GC1, DXY, EURUSD) → **%MoM** (roll-neutral return for futures)
- **without price** (USSP2/10/30 swap spreads) → **absolute Δ (bps)**

**Roll fix:** the front-contract roll calendar is taken from Bloomberg history (`FUT_CUR_GEN_TICKER`
sampled daily; a roll-month is where the actual contract changes). Deterministic contract-cycle
calendar as fallback. The old price-based roll heuristic is **removed** — it was manufacturing
fake seasonality. Cell 4b prints the detected roll frequency so you can sanity-check it.

One combined figure per asset (hist + seasonality bars + heatmap/YoY/seasonality-row), red=neg / green=pos,
PDF via `PdfPages`, then a joint VAR.

In [ ]:
import sys, subprocess
def _pip(*p): subprocess.run([sys.executable,'-m','pip','install','-q',*p], check=False)
_pip('matplotlib','statsmodels','pandas','numpy')

In [ ]:
import bql, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display
import warnings; warnings.filterwarnings('ignore')
bq = bql.Service()

In [ ]:
ASSETS = {
    'ES1 Index'    : dict(kind='future', label='ES1 (S&P500 fut)'),
    'VG1 Index'    : dict(kind='future', label='VG1 (EuroStoxx50 fut)'),
    'TY1 Comdty'   : dict(kind='future', label='TY1 (UST 10Y fut)'),
    'RX1 Comdty'   : dict(kind='future', label='RX1 (Bund fut)'),
    'DXY Curncy'   : dict(kind='fx',     label='DXY (USD index)'),
    'EURUSD Curncy': dict(kind='fx',     label='EURUSD (spot)'),
    'CL1 Comdty'   : dict(kind='future', label='CL1 (WTI fut)'),
    'GC1 Comdty'   : dict(kind='future', label='GC1 (Gold fut)'),
    'USSP2 Curncy' : dict(kind='spread', label='ASW2Y (2Y swap spread)'),
    'USSP10 Curncy': dict(kind='spread', label='ASW10Y (10Y swap spread)'),
    'USSP30 Curncy': dict(kind='spread', label='ASW30Y (30Y swap spread)'),
}
LOOKBACK = '-30Y'
MONTHS   = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
MPL_CMAP = 'RdYlGn'          # red = negative, green = positive
HEAT_FS, CELL_LW = 9, 3

CHUNK_YEARS = 5          # daily pulls are chunked to stay under the per-request cell cap
ROLL_DAY    = 'identity' # 'identity' = exact roll day taken from FUT_CUR_GEN_TICKER (the REAL
                         # historical roll calendar: the day the front contract actually changed).
                         # 'jump' (daily price-gap heuristic) is now an EXPLICIT opt-in only --
                         # it is NEVER a silent fallback (it manufactured fake seasonality).

CLIP_RET = 0.5          # daily |return| above this is treated as a data glitch (e.g. WTI Apr-2020 negative print)

# --- roll-calendar coverage control ----------------------------------------------------------------
# If FUT_CUR_GEN_TICKER history is SHORTER than the 30y price history, the early span has no real
# roll calendar. ROLL_FALLBACK decides what happens to that un-verifiable early span:
#   'restrict' -> drop it and warn (DEFAULT; never fabricates a roll date -> no fake seasonality)
#   'cycle'    -> extend backward with the deterministic ROLL_CYCLE below (approximate roll dates)
#   'none'     -> keep it unadjusted (legacy behaviour; roll gaps still present in the early span)
ROLL_FALLBACK = 'restrict'

# Per-future contract cycle: expected rolls/yr (for the sanity print) and, only if
# ROLL_FALLBACK=='cycle', the cycle months whose front-change defines a synthetic roll.
ROLL_CYCLE = {
    'ES1 Index' : dict(months=[3, 6, 9, 12],          per_yr=4,  label='quarterly HMUZ'),
    'VG1 Index' : dict(months=[3, 6, 9, 12],          per_yr=4,  label='quarterly HMUZ'),
    'TY1 Comdty': dict(months=[3, 6, 9, 12],          per_yr=4,  label='quarterly HMUZ'),
    'RX1 Comdty': dict(months=[3, 6, 9, 12],          per_yr=4,  label='quarterly HMUZ'),
    'CL1 Comdty': dict(months=list(range(1, 13)),     per_yr=12, label='monthly'),
    'GC1 Comdty': dict(months=[2, 4, 6, 8, 10, 12],   per_yr=6,  label='even-month GJMQVZ'),
}
ROLL_OUTLIER_K = 8.0    # a NON-roll day whose raw |ret| > K * 63d rolling-median => suspected MISSED roll
VERIFY_SPLICE  = False  # one-off: also pull the 2nd-generic identity to certify G1_new == G2_prev

# roll-inclusive continuous series -> roll handled at source, no stitching, never negative.
# Set an asset to a ready-made index to bypass generic stitching entirely:
ROLL_INCLUSIVE = {
    # 'CL1 Comdty': 'BCOMCL Index',    # <- uncomment to use the Bloomberg WTI subindex (roll baked in)
}

In [ ]:
# ===== ALL series retrieved DAILY (chunked) -> roll adjusted on the EXACT calendar day -> resampled to MONTHLY =====
# Snap-to-jump: FUT_CUR_GEN_TICKER flips on the LAST-TRADE day, but the generic PRICE rolls ~1 trading day
# later, so the identity date is NOT the price-jump day. snap_roll_to_jump() moves each roll onto the exact
# price-discontinuity day so no roll gap leaks. Safe when there is no lag (snaps to offset 0 = no change).
SNAP_ROLL        = True     # snap identity roll dates onto the true price-jump day
SNAP_LO, SNAP_HI = 0, 3    # search window (trading days) around each identity roll date
SNAP_THRESH      = 0.0025   # min roll-signature to move the date (else flat carry -> keep identity date)
def _chunks(step):
    y1 = pd.Timestamp.today().year; y0 = y1 - 30
    for a in range(y0, y1 + 1, step):
        yield f'{a}-01-01', f'{min(a+step-1, y1)}-12-31'

def _fetch_daily(universe, item_fn, alias='v'):
    parts = []
    for s, e in _chunks(CHUNK_YEARS):
        df = bq.execute(bql.Request(universe, {alias: item_fn(bq.func.range(s, e))}))[0].df().reset_index()
        dc = next(c for c in df.columns if 'DATE' in c.upper())
        df = df.rename(columns={dc: 'DATE'}); df['DATE'] = pd.to_datetime(df['DATE']); parts.append(df)
    df = pd.concat(parts, ignore_index=True); idc = df.columns[0]
    return df.drop_duplicates([idc, 'DATE']).sort_values([idc, 'DATE']), idc

def px_daily(universe):
    df, idc = _fetch_daily(universe, lambda dr: bq.data.px_last(dates=dr, frq='D', fill='prev'))
    return {sec: g.set_index('DATE')['v'].astype(float).sort_index() for sec, g in df.groupby(idc)}

def cid_daily(ticker):
    # daily front-contract identity -> the exact roll DAY is where it changes
    try:
        df, _ = _fetch_daily(ticker, lambda dr: bq.data.fut_cur_gen_ticker(dates=dr, frq='D', fill='prev'))
        s = df.set_index('DATE')['v'].sort_index().dropna()
        return s if s.nunique() > 1 else None
    except Exception as ex:
        print('    daily identity unavailable:', str(ex).splitlines()[0][:60]); return None

def _root(t): return t.split()[0][:-1]
def _gen2(t): root, sec = t.split(); return f'{root[:-1]}2 {sec}'

# --- OPT-IN ONLY (ROLL_DAY='jump'): daily price-gap heuristic. NEVER a silent fallback anymore. -----
def roll_days(ticker, f1, f2):
    same  = (np.log(f1) - np.log(f1.shift(1))).abs()
    cross = (np.log(f1) - np.log(f2.shift(1))).abs()
    thr   = same.rolling(90, min_periods=20).median() * 4
    return ((cross < same * 0.5) & (same > thr) & f2.shift(1).notna()), 'jump'

def to_month_end(daily):
    m = daily.resample('M').last(); m.index = m.index.to_period('M'); return m

# --- explicit historical roll CALENDAR (the actual dates the front contract changed) ---------------
def _cid_series(ticker):
    """Daily front-generic underlying-contract identity, cleaned; None if unavailable/flat."""
    if ROLL_DAY != 'identity':
        return None
    s = cid_daily(ticker)                                # dropna'd; None if <2 distinct values or on error
    if s is None:
        return None
    return s[~s.index.duplicated(keep='last')].sort_index()

def roll_calendar(ticker, price_index):
    """Roll dates mapped EXACTLY onto the price trading-day grid.
    A roll date = first price day on/after the day FUT_CUR_GEN_TICKER changes contract (no ffill,
    so the flag can never smear onto a stale/phantom day).
    Returns (roll_dates: DatetimeIndex, covered_start: Timestamp, cid) or (None, None, None)."""
    cid = _cid_series(ticker)
    if cid is None:
        return None, None, None
    changed = cid.ne(cid.shift(1)) & cid.shift(1).notna()   # drops the leading NaN -> first-value edge
    change_dates = pd.DatetimeIndex(cid.index[changed])
    pi = pd.DatetimeIndex(price_index).sort_values().unique()
    if len(change_dates) == 0 or len(pi) == 0:
        return pd.DatetimeIndex([]), cid.index.min(), cid
    pos = pi.searchsorted(change_dates, side='left')       # first price day >= each change date
    pos = np.unique(pos[pos < len(pi)])
    return pi[pos], cid.index.min(), cid

def snap_roll_to_jump(roll_dates, f1, f2, lo=SNAP_LO, hi=SNAP_HI, thresh=SNAP_THRESH, min_support=0.4):
    """Snap identity roll dates onto the true price-discontinuity day (modal, two-pass).
    Roll signature at day j: sig = |f1[j]/f1[j-1]-1| - |f1[j]/f2[j-1]-1|, which peaks where switching to
    the cross-contract return collapses a jump (the real roll day). Pass 1 finds each roll's best offset
    from clearly-signed rolls; because the roll convention is CONSISTENT we take the MODAL offset and, if
    it is well-supported, (pass 2) snap EVERY roll to that offset wherever prices are valid -- even rolls
    whose local signature is obscured because that day's market move opposed the roll gap. Falls back to
    the per-roll best, then the identity date. Returns (snapped DatetimeIndex, {snapped_date: j-i offset})."""
    idx = f1.index; n = len(idx)
    if roll_dates is None or len(roll_dates) == 0 or n < 2:
        return pd.DatetimeIndex([]), {}
    f1v = f1.to_numpy(dtype=float); f2v = f2.to_numpy(dtype=float)
    pos = [int(i) for i in idx.get_indexer(pd.DatetimeIndex(roll_dates)) if i >= 0]
    if not pos:
        return pd.DatetimeIndex([]), {}

    def jump_at(j):                                                # (same, cross, sig) or None if unusable
        if j < 1 or j >= n:
            return None
        a, b, c = f1v[j], f1v[j - 1], f2v[j - 1]
        if not (np.isfinite(a) and np.isfinite(b) and np.isfinite(c) and a > 0 and b > 0 and c > 0):
            return None
        same = abs(a / b - 1.0); cross = abs(a / c - 1.0)
        return same, cross, same - cross

    def is_roll(s):                                                # a genuine, clearly-signed roll day
        return s is not None and s[2] > thresh and s[1] < 0.5 * s[0]

    # --- pass 1: best offset for each clearly-signed roll ------------------------------------------
    best = {}
    for i in pos:
        bj, bs = None, -np.inf
        for j in range(max(i + lo, 1), min(i + hi, n - 1) + 1):
            s = jump_at(j)
            if is_roll(s) and s[2] > bs:
                bs, bj = s[2], j
        if bj is not None:
            best[i] = bj - i
    from collections import Counter
    o_star  = Counter(best.values()).most_common(1)[0][0] if best else 0     # modal (consistent) lag
    support = sum(v == o_star for v in best.values()) / len(pos)             # fraction of rolls agreeing

    # --- pass 2: trust the modal offset for ALL rolls when well-supported; else per-roll best; else id
    snapped_pos, offsets = [], {}
    for i in pos:
        cand = i + o_star
        if support >= min_support and cand >= 1 and jump_at(cand) is not None:
            j_star = cand                                          # consistent convention -> snap all rolls
        elif i in best and i + best[i] >= 1:
            j_star = i + best[i]
        elif i >= 1:
            j_star = i                                             # keep identity date (flat carry / no jump)
        else:
            continue
        snapped_pos.append(j_star); offsets[idx[j_star]] = int(j_star - i)
    if not snapped_pos:
        return pd.DatetimeIndex([]), {}
    uniq = np.unique(np.asarray(snapped_pos, dtype=int))           # dedup collisions + sort
    return pd.DatetimeIndex(idx[uniq]), {idx[p]: offsets[idx[p]] for p in uniq}

def _cycle_roll_dates(ticker, price_index, start, end):
    """OPT-IN (ROLL_FALLBACK='cycle'): deterministic synthetic roll dates from the contract cycle
    for [start, end). Approximate (anchored to the cycle month) -- validate before trusting."""
    cyc = ROLL_CYCLE.get(ticker)
    if cyc is None:
        return pd.DatetimeIndex([])
    pi, out = pd.DatetimeIndex(price_index), []
    for yr in range(start.year, end.year + 1):
        for mo in cyc['months']:
            j = pi.searchsorted(pd.Timestamp(yr, mo, 1), side='left')
            if 0 <= j < len(pi) and start <= pi[j] < end:
                out.append(pi[j])
    return pd.DatetimeIndex(sorted(set(out)))

def _roll_diagnostics(ticker, f1, roll_mask, roll_dates, n_bad, snap_offsets=None, r=None):
    """Per-roll diagnostics: roll rate, snap-offset histogram (the lag fingerprint), mean roll gap
    captured, residual raw-jump outliers, and a SYSTEMATIC roll-leak permutation test on the daily
    returns (catches the small, repeating, calendar-aligned leak the 8x-median filter is blind to)."""
    from collections import Counter
    raw = (f1 / f1.shift(1) - 1.0).abs()
    thr = raw.rolling(63, min_periods=20).median() * ROLL_OUTLIER_K
    unflagged_big = (raw > thr) & (~roll_mask) & f1.shift(1).gt(0) & f1.gt(0)
    resid = f1.index[unflagged_big.fillna(False).values]
    yrs   = max(f1.index.year.max() - f1.index.year.min(), 1)
    exp   = ROLL_CYCLE.get(ticker, {}).get('per_yr')
    exp_s = f' (expect ~{exp}/yr)' if exp else ''
    flag  = f' guarded={n_bad}' if n_bad else ''
    print(f'    {ticker:12s} rolls={len(roll_dates):4d} ~{len(roll_dates)/yrs:4.1f}/yr{exp_s}{flag}  '
          f'residual-unflagged-outliers={len(resid)}')
    if snap_offsets:
        dist = dict(sorted(Counter(snap_offsets.values()).items()))
        f1s  = f1.shift(1)
        gaps = [abs(f1.loc[d] / f1s.loc[d] - 1.0) for d in roll_dates
                if d in f1.index and pd.notna(f1s.loc[d]) and f1s.loc[d] > 0]
        mg = 100 * np.mean(gaps) if gaps else float('nan')
        print(f'       snap offsets (price-jump lag vs identity date): {dist}  mean |roll gap| captured: {mg:.2f}%')
    if r is not None and len(roll_dates):
        rm = r.index.isin(pd.DatetimeIndex(roll_dates))
        roll_abs = r[rm].abs().dropna(); pool = r[~rm].abs().dropna().to_numpy()
        k = len(roll_abs)
        if k and len(pool):
            obs = roll_abs.mean(); rng = np.random.default_rng(0)
            boot = np.array([rng.choice(pool, k, replace=(len(pool) < k)).mean() for _ in range(2000)])
            p = float((boot >= obs).mean()); lk = 'LEAK' if p < 0.05 else 'ok'
            print(f'       roll-leak test: roll-day mean|r|={obs*100:.3f}% vs random={boot.mean()*100:.3f}%  p={p:.3f} [{lk}]')
    if len(resid):
        shown = [d.date().isoformat() for d in resid[:8]]
        print(f'       !! residual raw jumps (non-roll): {shown}' + (' ...' if len(resid) > 8 else ''))

def _verify_splice_mapping(ticker, roll_dates, price_index):
    """OPT-IN (VERIFY_SPLICE): certify G1_new == G2_prev on roll days (generic cycle-skip check)."""
    c1, c2 = _cid_series(ticker), _cid_series(_gen2(ticker))
    if c1 is None or c2 is None:
        print(f'    {ticker:12s} splice-mapping check skipped (identity missing)'); return pd.DatetimeIndex([])
    c1 = c1.reindex(price_index).ffill(); c2 = c2.reindex(price_index).ffill()
    bad = []
    for d in roll_dates:
        i = price_index.get_indexer([d])[0]
        if i > 0 and c1.iloc[i] != c2.iloc[i - 1]:
            bad.append(d)
    print(f'    {ticker:12s} G1_new==G2_prev  ok={len(roll_dates) - len(bad)}  mismatch={len(bad)}')
    if bad:
        print(f'       generic-skip roll days: {[x.date().isoformat() for x in bad[:6]]}')
    return pd.DatetimeIndex(bad)

def future_mom(ticker):
    # roll-inclusive index override -> roll handled at source, no stitching
    if ticker in ROLL_INCLUSIVE:
        src = ROLL_INCLUSIVE[ticker]
        s = to_month_end(px_daily([src])[src])
        print(f'    {ticker:12s} using roll-inclusive series {src} (roll baked in)')
        return (s.pct_change() * 100).dropna()

    g1, g2 = ticker, _gen2(ticker)
    px = px_daily([g1, g2])
    f1 = px[g1].astype(float).sort_index()
    f2 = px.get(g2)
    if f2 is None:                                          # no 2nd generic -> cannot splice
        print(f'    !! {ticker:12s} 2nd generic {g2} unavailable -> UNADJUSTED (roll gaps present)')
        ROLLCAL[ticker] = pd.DatetimeIndex([]); ROLLCOV[ticker] = (f1.index.min(), f1.index.max(), None)
        return (to_month_end(f1).pct_change() * 100).dropna()
    f2 = f2.astype(float).reindex(f1.index)                 # align onto f1's NATIVE grid (no union/ffill smear)

    roll_dates, covered_start, cid = roll_calendar(ticker, f1.index)

    # --- resolve the roll calendar / coverage --------------------------------------------------------
    if roll_dates is None:                                  # identity unavailable -> NO silent jump fallback
        if ROLL_DAY == 'jump':                              # explicit opt-in only
            rm, _ = roll_days(ticker, f1, f2.reindex(f1.index).ffill())
            roll_dates = f1.index[rm.reindex(f1.index).fillna(False).astype(bool)]
            covered_start = f1.index.min()
            print(f'    !! {ticker:12s} identity unavailable -> EXPLICIT price-jump heuristic (approximate)')
        elif ROLL_FALLBACK == 'cycle':
            print(f'    !! {ticker:12s} identity unavailable -> deterministic cycle calendar (UNVALIDATED)')
            roll_dates = _cycle_roll_dates(ticker, f1.index, f1.index.min(), f1.index.max())
            covered_start = f1.index.min()
        else:
            print(f'    !! {ticker:12s} identity unavailable -> UNADJUSTED (honest; roll gaps present)')
            ROLLCAL[ticker] = pd.DatetimeIndex([]); ROLLCOV[ticker] = (f1.index.min(), f1.index.max(), None)
            return (to_month_end(f1).pct_change() * 100).dropna()

    # coverage gap: identity history shorter than price history
    price_start = f1.index.min()
    if covered_start is not None and covered_start > price_start:
        gap = (covered_start - price_start).days / 365.25
        if ROLL_FALLBACK == 'cycle':
            extra = _cycle_roll_dates(ticker, f1.index, price_start, covered_start)
            roll_dates = pd.DatetimeIndex(roll_dates).union(extra); eff_start = price_start
            print(f'    {ticker:12s} identity from {covered_start.date()}; +{len(extra)} synthetic cycle '
                  f'rolls over the prior {gap:.1f}y')
        elif ROLL_FALLBACK == 'restrict':
            eff_start = covered_start
            print(f'    {ticker:12s} identity from {covered_start.date()}; RESTRICTING output '
                  f'(dropping {gap:.1f}y with no real roll calendar)')
        else:                                              # 'none' -> keep early span unadjusted
            eff_start = price_start
            print(f'    {ticker:12s} identity from {covered_start.date()}; keeping {gap:.1f}y early span '
                  f'UNADJUSTED (roll gaps present)')
    else:
        eff_start = price_start

    # slice everything to the effective window
    f1 = f1[f1.index >= eff_start]; f2 = f2[f2.index >= eff_start]
    roll_dates = pd.DatetimeIndex(roll_dates)
    roll_dates = roll_dates[roll_dates >= eff_start]
    snap_offsets = {}
    if SNAP_ROLL and len(roll_dates):                       # snap each roll onto the true price-jump day
        roll_dates, snap_offsets = snap_roll_to_jump(roll_dates, f1, f2)
    roll_mask = pd.Series(f1.index.isin(roll_dates), index=f1.index)

    if VERIFY_SPLICE and len(roll_dates):
        skip = _verify_splice_mapping(ticker, roll_dates, f1.index)
        if len(skip):
            roll_mask &= ~pd.Series(f1.index.isin(skip), index=f1.index)   # drop skip-tainted splices

    # --- ratio splice on EXACTLY the calendar dates + denominator-aware guard -------------------------
    r_normal = f1 / f1.shift(1) - 1.0                      # same-contract daily return
    r_roll   = f1 / f2.shift(1) - 1.0                      # roll day: new front today / new front yesterday (=2nd gen)
    r        = r_normal.where(~roll_mask, r_roll)
    den      = f1.shift(1).where(~roll_mask, f2.shift(1))  # the denominator actually used that day
    bad      = (f1 <= 0) | (den <= 0) | (r.abs() > CLIP_RET)
    n_bad    = int(bad.sum())
    r        = r.mask(bad, np.nan)

    lvl = (1 + r.fillna(0)).cumprod()                      # clean daily roll-adjusted level
    m   = to_month_end(lvl)                                # DAILY -> MONTHLY (month-end of clean level)

    ROLLCAL[ticker] = roll_dates
    ROLLCOV[ticker] = (eff_start, f1.index.max(), covered_start)
    SNAP_REPORT[ticker] = snap_offsets
    _roll_diagnostics(ticker, f1, roll_mask, roll_dates, n_bad, snap_offsets, r)
    return (m.pct_change() * 100).dropna()

def series_for(ticker, cfg):
    if cfg['kind'] == 'future':
        r = future_mom(ticker); lvl = (1 + r/100).cumprod()
        return r, lvl.pct_change(12) * 100, '%MoM'
    s = to_month_end(px_daily([ticker])[ticker])            # fx / spread: daily -> month-end
    if cfg['kind'] == 'fx':
        return (s.pct_change() * 100).dropna(), s.pct_change(12) * 100, '%MoM'
    return s.diff().dropna(), s.diff(12), 'Δ bps'           # spread: absolute variation (bps)

MET, YOY, MLAB = {}, {}, {}
ROLLCAL, ROLLCOV, SNAP_REPORT = {}, {}, {}      # per-future roll calendar (dates) + coverage window, filled by future_mom
print(f'ROLL_DAY={ROLL_DAY}  ROLL_FALLBACK={ROLL_FALLBACK}. '
      f'roll-days check (expect ~4/yr ES/VG/TY/RX, ~12/yr CL, ~6/yr GC):')
for tk, cfg in ASSETS.items():
    try:
        m, y, lab = series_for(tk, cfg)
        MET[tk], YOY[tk], MLAB[tk] = m.dropna(), y, lab
        if cfg['kind'] != 'future':
            print(f'    {tk:12s} n={MET[tk].shape[0]}  ({lab})')
    except Exception as e:
        print(f'ERR {tk:15s} {str(e).splitlines()[0][:110]}')

def _panels(ticker):
    s = MET[ticker]
    df = pd.DataFrame({'v': s.values}, index=s.index)
    df['year'], df['month'] = df.index.year, df.index.month
    pivot = df.pivot_table(index='year', columns='month', values='v').reindex(columns=range(1,13))
    seas  = df.groupby('month')['v'].mean().reindex(range(1,13))
    y = YOY[ticker]; yoy = y.groupby(y.index.year).last().reindex(pivot.index)
    return pivot, seas, yoy

In [ ]:
# ===== 4b: the roll calendar actually used (the historical roll-down dates, per future) =====
# One block per future: total rolls, the effective window, the date FUT_CUR_GEN_TICKER coverage
# starts, per-year roll counts (CL should be ~12/yr in EVERY year, ES/VG/TY/RX ~4, GC ~6), and
# the first dates so you can eyeball that rolls land on real roll days, not month-ends.
print('Roll calendars (from FUT_CUR_GEN_TICKER, mapped to the price trading-day grid):')
for tk in ASSETS:
    if tk not in ROLLCAL:
        continue
    rd = pd.DatetimeIndex(ROLLCAL[tk])
    eff, last, cov = ROLLCOV[tk]
    cov_s = pd.Timestamp(cov).date().isoformat() if cov is not None else 'n/a'
    if len(rd) == 0:
        print(f'  {tk:12s} (no rolls detected) | identity-from {cov_s}')
        continue
    per_yr = pd.Series(1, index=rd).groupby(rd.year).size()
    print(f'  {tk:12s} {len(rd):4d} rolls | window {pd.Timestamp(eff).date()} -> {pd.Timestamp(last).date()} '
          f'| identity-from {cov_s}')
    print(f'     per-year counts: {per_yr.to_dict()}')
    print(f'     first roll dates: {[d.date().isoformat() for d in rd[:12]]}'
          + (' ...' if len(rd) > 12 else ''))

In [ ]:
def _pcm(ax, M, xl, yl, vlim, fs=HEAT_FS, lw=CELL_LW):
    cmap = plt.get_cmap(MPL_CMAP).copy(); cmap.set_bad('white')
    norm = TwoSlopeNorm(vmin=-vlim, vcenter=0.0, vmax=vlim)
    ax.pcolormesh(np.ma.masked_invalid(M), cmap=cmap, norm=norm, edgecolors='white', linewidth=lw)
    ax.set_xticks(np.arange(len(xl)) + 0.5); ax.set_xticklabels(xl, fontsize=fs + 1)
    ax.set_yticks(np.arange(len(yl)) + 0.5); ax.set_yticklabels(yl, fontsize=fs)
    ax.invert_yaxis()
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            if np.isfinite(v):
                ax.text(j + 0.5, i + 0.5, f'{v:.2f}', ha='center', va='center', fontsize=fs)

def make_zone_figure(ticker):
    name = ASSETS[ticker]['label']; s = MET[ticker]; lab = MLAB[ticker]
    pivot, seas, yoy = _panels(ticker)
    years = pivot.index.astype(str).tolist(); n = len(years)
    vlim = max(np.nanpercentile(np.abs(pivot.values), 98), 0.01)
    fin = np.isfinite(yoy.values)
    yv = max(np.nanpercentile(np.abs(yoy.values[fin]), 98), 0.01) if fin.any() else 1.0

    fig = plt.figure(figsize=(12, max(9.0, 0.32 * n + 5.0)))
    master = fig.add_gridspec(2, 1, height_ratios=[1.0, 2.6], hspace=0.30)
    gs_top = master[0].subgridspec(1, 2, wspace=0.22)
    gs_bot = master[1].subgridspec(2, 2, width_ratios=[0.92, 0.08],
                                   height_ratios=[0.93, 0.07], wspace=0.04, hspace=0.06)

    axH = fig.add_subplot(gs_top[0, 0])
    axH.hist(s.values, bins=40, color='seagreen', edgecolor='white')
    axH.axvline(s.mean(), color='crimson', ls='--', lw=1.2, label=f'mean {s.mean():.2f}')
    axH.set_title(f'{name}: {lab} distribution'); axH.set_xlabel(lab); axH.set_ylabel('count'); axH.legend()

    axS = fig.add_subplot(gs_top[0, 1])
    axS.bar(MONTHS, seas.values, color=['seagreen' if v >= 0 else 'crimson' for v in seas.values],
            edgecolor='white')
    axS.axhline(0, color='k', lw=0.6)
    for i, v in enumerate(seas.values):
        if np.isfinite(v):
            axS.text(i, v, f'{v:.2f}', ha='center', va='bottom' if v >= 0 else 'top', fontsize=8)
    axS.set_title(f'avg {lab} by month (seasonality)'); axS.set_ylabel(lab)

    axM = fig.add_subplot(gs_bot[0, 0]); axY = fig.add_subplot(gs_bot[0, 1]); axR = fig.add_subplot(gs_bot[1, 0])
    _pcm(axM, pivot.values, MONTHS, years, vlim); axM.set_xticklabels([])
    _pcm(axY, yoy.values.reshape(-1, 1), ['YoY'], years, yv); axY.set_yticklabels([])
    _pcm(axR, seas.values.reshape(1, -1), MONTHS, ['Avg'], vlim)
    axM.set_title(f'{lab} heatmap (year×month)  |  YoY vector (right)  |  seasonality row (below)', fontsize=10)

    fig.suptitle(f'{ticker} — {name}', fontsize=14, y=0.995)
    return fig

## Combined figures + PDF report (1 asset/page) via PdfPages

In [ ]:
def build_report(path='crossasset_seasonality.pdf', show=True):
    with PdfPages(path) as pdf:
        for t in MET:
            fig = make_zone_figure(t)
            pdf.savefig(fig, bbox_inches='tight')
            if show: display(fig)
            plt.close(fig)
    print('Saved', os.path.abspath(path))

build_report()

## VAR on the joint metric panel
Monthly dummies (`exog`) absorb seasonality; lag by AIC. Units mixed (%MoM vs Δbps) → read signs / relative dynamics.

In [ ]:
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller

panel = pd.concat({ASSETS[t]['label']: MET[t] for t in MET}, axis=1).dropna()
panel.index = panel.index.to_timestamp()
print('Panel:', panel.shape, '|', panel.index.min().date(), '->', panel.index.max().date())

print('\nADF (H0: unit root):')
for c in panel.columns:
    p = adfuller(panel[c].dropna(), autolag='AIC')[1]
    print(f'  {c:24s} p={p:.2f}  {"stationary" if p < 0.05 else "NON-stationary"}')

seas_exog = pd.get_dummies(panel.index.month, prefix='m', drop_first=True)
seas_exog.index = panel.index; seas_exog = seas_exog.astype(float)

In [ ]:
model = VAR(panel, exog=seas_exog)
sel   = model.select_order(maxlags=12)
print(sel.summary())
lag = int(sel.selected_orders['aic']) or 1
res = model.fit(lag)
print(f'\nFitted VAR(p={lag}) + monthly dummies\n'); print(res.summary())

In [ ]:
res.irf(12).plot(orth=True); plt.tight_layout(); plt.show()

In [ ]:
res.fevd(12).plot(); plt.tight_layout(); plt.show()